序列方面的建模

# 一, 参考DeepPCT

## ⚠️比对来源：

* src/feature/sequence_embed.py, 


* DeepPCT/blob/main/features/sequence_embedding.py

## 1, ESM-2的Embedding: 本质即Encoder

Encoder (ESM): 预训练模型提取特征（这一步自带了极强的 Self-Attention 上下文）

### 核心差异与修改解析

目前的任务是从**DeepPCT（单一序列内部残基对互作）**迁移到**PPI（双序列间互作）**。在序列建模（Sequence Embedding）层面，存在以下关键区别：

#### 1. 建模对象不同
*   **DeepPCT**: 关注同一个蛋白内的**位点对 (Site Pair)**。
    *   **Embed Logic**: 提取位点 $i$ 和 $j$ 周围的**局部窗口 (Window)** embedding。
    *   **Relation**: 直接使用 ESM-2 的 **Attention Map (自注意力图)** 作为特征，因为它天生描述了同一序列内 $i$ 和 $j$ 的关系。
*   **我的 IDR-PPI**: 关注序列 A 与序列 B 的**全局或区域互作**。
    *   **Embed Modification**: IDR通常作为一个整体功能区域，且长短不一，不仅限于局部窗口。因此，我们提取**全序列 (Full Sequence)** 的 embedding，保留所有残基特征供后续 Cross-Attention 使用。
    *   **Relation**: 由于是对 A 和 B 分别提取 embedding，ESM-2 无法直接提供 A-B 间的 Attention Map。因此，我们需要在 `model.py` 中设计 `CrossAttentionLayer` ，它“手动”构建了 A 和 B 之间的交互注意力，替代了 DeepPCT 中 ESM 自带的 Pair Attention。



```mermaid
graph TD
    %% 定义样式
    style Input fill:#f9f,stroke:#333,stroke-width:2px
    style ESM fill:#eee,stroke:#333,stroke-dasharray: 5 5
    style Region fill:#bbf,stroke:#333
    style CrossAttn fill:#fdb,stroke:#333
    style Pool fill:#dfd,stroke:#333
    style Classifier fill:#ffd,stroke:#333

    subgraph Inputs
        IA[Seq A] --> IA_Str["Protein A String"]
        IB[Seq B] --> IB_Str["Protein B String"]
        RA[Region A]
        RB[Region B]
    end

    subgraph "1. Encoder (Frozen ESM-2)"
        IA_Str --> ESM_A[ESM-2 Model]
        IB_Str --> ESM_B[ESM-2 Model]
    end

    subgraph "2. Region Extraction"
        ESM_A --"Full Embed A"--> Extract_A["Extract & Pad Region A"]
        RA --> Extract_A
        ESM_B --"Full Embed B"--> Extract_B["Extract & Pad Region B"]
        RB --> Extract_B
        
        Extract_A --"Feat A (Query)"--> FeatA[Region Embed A]
        Extract_B --"Feat B (Key/Val)"--> FeatB[Region Embed B]
    end

    subgraph "3. Interaction (Cross-Attention)"
        %% A to B
        FeatA --Query--> CA_A2B[Cross Attn A2B]
        FeatB --Key/Val--> CA_A2B
        CA_A2B --> InterA[Interacted A]
        
        %% B to A
        FeatB --Query--> CA_B2A[Cross Attn B2A]
        FeatA --Key/Val--> CA_B2A
        CA_B2A --> InterB[Interacted B]
    end

    subgraph "4. Aggregation"
        FeatA --> PoolA[Avg Pool]
        FeatB --> PoolB[Avg Pool]
        InterA --> Pool_IA[Avg Pool]
        InterB --> Pool_IB[Avg Pool]
        
        PoolA --> Concat[Check Concatenate]
        PoolB --> Concat
        Pool_IA --> Concat
        Pool_IB --> Concat
    end

    subgraph "5. Prediction"
        Concat --"[Batch, 1280*4]"--> MLP[MLP Classifier]
        MLP --> Logits["Logits (Prob)"]
    end
```

## 2, 除了Encoder之外的部分？

### 1, 为什么不用标准Transformer的Decoder部分？

这是一个非常好的架构对比问题。

首先：**我们的 `DeepIDRSeq` 其实已经使用了 Transformer Decoder 的核心组件，但为了适应这一特定任务（PPI 互作），它去掉了对于该任务无用的“Masked Self-Attention”部分。**

`DeepPCTseq`（你提供的代码）和 `DeepIDRSeq`（当前工作区的代码）在架构设计上有显著的区别。当前的 `DeepIDRSeq` 设计更加完整和标准化（Closer to standard Transformer definition）。

### 1. 为什么不用标准的 Transformer Decoder？

一个标准的 Transformer Decoder Block 包含三个子层：
1.  **Masked Self-Attention**: 用于生成任务（如GPT），防止看到未来的词。
2.  **Cross-Attention**: 查询（Query）关注 编码器输出（Key/Value）。
3.  **Feed Forward Network (FFN)**: 全连接层 + 激活函数。

**在你的任务（PPI 互作预测）中：**
*   **Masked Self-Attention 是累赘**：你不需要生成序列（不需要 predict next token），你已经有了完整的蛋白质 A 和 蛋白质 B 的序列。因此，不需要 Masked Self-Attention。
*   **只需要 Cross-Attention**：我们需要的是“让 A 看到 B”以及“让 B 看到 A”。

所以，你的 `DeepIDRSeq` 中的 `TransformerWebBlock` **实际上就是一个精简版的 Decoder Block**（保留了 Cross-Attn 和 FFN，去掉了 Masked Self-Attn）。这是判别式模型（Discriminative Model）处理双塔交互的标准做法。

### 2. DeepIDRSeq (你的代码) vs DeepPCTseq (参考代码) 的深度对比

这两个模型的架构**完全不一样**。你的 `DeepIDRSeq` 更强、更深、更符合 Transformer 的设计哲学。

#### 架构差异图解

| 特性 | DeepIDRSeq (当前代码) | DeepPCTseq (参考代码) | 评价 |
| :--- | :--- | :--- | :--- |
| **基础单元** | **Full Block** (Attn + Norm + **FFN**) | **Look-Only** (Attn + Norm) | **DeepIDRSeq 胜**。缺少 FFN (Feed Forward) 意味着模型失去了非线性变换能力，只能做简单的特征加权。 |
| **交互方向** | **双向** (A→B 且 B→A) | **单向** (Window2 → Window1) | **DeepIDRSeq 胜**。PPI 是相互的，A 影响 B，B 也影响 A。DeepPCTseq 只有单向查询。 |
| **输入范围** | **变长区域** (整个 Region) | **定长窗口** (固定 11+11 氨基酸) | **DeepIDRSeq 灵活性更高**。能处理任意长度的 IDR。 |
| **特征融合** | Concat(Pool(A), Pool(B), Pool(A2B), Pool(B2A)) | Concat(Center(A2B), Pair_Embed) | DeepIDRSeq 保留了更多上下文信息。 |

### 3. DeepPCTseq 代码解析 (为何它比较简单)

你提供的 `DeepPCTseq` 代码是一个非常简化的版本：



In [ ]:
class CrossAttentionMutilLayer(nn.Module):
    # ...
    def forward(self, tgt, memory):
        for layer in self.cross_attention_layers:
            # 只有 Attention，没有 FFN！
            x, _ = layer(query=tgt, key=memory, value=memory, need_weights=False)
            tgt = self.layer_norm(tgt + x) 
        return tgt



1.  **缺少 FFN**: 它做完 Attention 后直接 Norm 就输出了。在现代深度学习中，FFN（通常是两层 MLP + GELU）负责处理特征的非线性映射，非常重要。缺少它会限制模型的表达能力。
2.  **局部视角**: 它的输入是 `residue_embedding_window`，切分成 `x_window_1` 和 `x_window_2`。这看起来像是在看两个具体的残基位点及其邻域。
3.  **依赖 Pair Embedding**: 它非常依赖 `residue_pair_embedding`（可能是距离矩阵或接触图），而 `DeepIDRSeq` 试图纯粹从序列和 Cross-Attention 中学习交互。

### 4.  DeepIDRSeq 架构流

`model.py` 中的架构是非常稳健的 **Siamese Interaction** 架构：

1.  **Encoder (ESM)**: 预训练模型提取特征（这一步自带了极强的 Self-Attention 上下文）。
2.  **Interaction (X-Attn)**:
    *   `TransformerWebBlock` = Standard Attention + **FFN** (这很重要)。
    *   逻辑流：`A = A + Attn(A, B)` -> `A = A + FFN(A)`。
3.  **Symmetry (对称性)**:
    *   计算 `A_context = Attend(A, B)`
    *   计算 `B_context = Attend(B, A)`
    *   最后拼起来。

**总结**：
这里的 `CrossAttentionModule` **就是** 针对 PPI 任务定制的 Decoder。它比你贴出来的 `DeepPCTseq` 更高级，因为它保留了 Transformer 必需的 FFN 层，并且设计了双向交互，能够捕捉更复杂的蛋白质-蛋白质相互作用模式。

# 二, 获取数据

In [13]:
import requests, time
def get_Uniprot_protein_fasta(query: str, format: str, compressed: bool, size: int) -> list[str]:
    """
    Description:
        根据给定的查询参数, 从Uniprot数据库中获取蛋白质的fasta序列, 支持分页请求;
        
    Args:
        query (str): 查询字符串, 用于指定搜索条件, 可以是单个序列数据， 也可以是批量查询;
        format (str): 返回数据的格式, 默认为"fasta";
        compressed (bool): 是否请求压缩格式的数据, 默认为False;
        size (int): 每页返回的记录数/单次请求返回的序列数, 默认为500
    
    Returns:
        list[str]: 包含所有获取到的fasta序列的列表
    """

    base = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": query,
        "format": format,
        "compressed": compressed,
        "size": size
    }
    # 用于存储分页请求返回的fasta序列片段
    fasta_parts = []
    url = base
    # 分页请求循环，直到没有下一页
    while True:
        # 首次请求时使用params字典，后续请求的"下一页"url已包含所有必要参数，因此不再重复传入params参数
        # 请求超时设置为60秒
        r = requests.get(url, params=params if url == base else None, timeout=60)
        # 检查是否请求成功，否则抛出异常
        r.raise_for_status()
        
        # 解析响应内容，将当前请求返回的fasta序列添加到列表中
        fasta_parts.append(r.text)

        # 解析下一页url
        # 先尝试获取 “next” 对应的子字典，若不存在则返回空字典；再从子字典中获取 “url”，若不存在则返回None（表示无下一页）
        next_url = r.links.get("next", {}).get("url")
        # 若无下一页，则跳出循环
        if not next_url:
            break
        # 更新url为下一页的url，继续请求
        # next_url已包含所有参数，无需重复传递
        url, params = next_url, None
        # 为避免过于频繁请求，稍作延时
        time.sleep(0.2)

    # 返回所有获取到的fasta序列片段, 拼接为一个完整的字符串
    fasta_str = "".join(fasta_parts)
    # 最好分割为一个accession对应一个字符串的列表返回
    fasta_list = fasta_str.strip().split("\n>")

    # 假设我们都是检索一条序列
    fasta_entry = fasta_list[0]
    # 然后分割每一条序列
    fasta_seq = "".join(fasta_entry.split("\n")[1:])

    return fasta_seq	




In [14]:
result = get_Uniprot_protein_fasta(query="P49711", format="fasta", compressed=False, size=500)

```mermaid
graph TD
    %% 定义样式
    classDef input fill:#f9f,stroke:#333,stroke-width:2px;
    classDef encoder fill:#eee,stroke:#333,stroke-width:2px;
    classDef process fill:#d4e1f5,stroke:#333,stroke-width:2px;
    classDef decoder fill:#fff2cc,stroke:#d6b656,stroke-width:2px,stroke-dasharray: 5 5;
    classDef head fill:#e1d5e7,stroke:#9673a6,stroke-width:2px;

    %% 1. 输入层
    subgraph Input_Layer [输入层]
        InA("Input Sequence A")
        InB("Input Sequence B")
    end

    %% 2. 编码层 (Siamese ESM-2)
    subgraph Encoder_Layer ["预训练特征提取 (Frozen)"]
        ESM("Shared ESM-2 Backbone")
    end

    InA --> ESM
    InB --> ESM

    %% 3. 区域提取
    subgraph Region_Layer [区域特征提取]
        ExtractA["Region Extraction A"]
        ExtractB["Region Extraction B"]
    end

    ESM -- "Full Embed A" --> ExtractA
    ESM -- "Full Embed B" --> ExtractB

    %% 4. 双向解码层 (核心)
    subgraph Interaction_Layer ["双向交互解码器 (Inter-Protein Decoder)"]
        direction LR
        
        %% Decoder A2B
        subgraph Dec_A2B ["Decoder A2B: A attends to B"]
            CA1["Cross-Attention<br>Q=A, K=B, V=B"]
            FFN1["Feed Forward Network"]
        end

        %% Decoder B2A
        subgraph Dec_B2A ["Decoder B2A: B attends to A"]
            CA2["Cross-Attention<br>Q=B, K=A, V=A"]
            FFN2["Feed Forward Network"]
        end
    end

    %% 连接 Encoder 到 Decoder
    ExtractA -->|Query| CA1
    ExtractB -->|"Key / Value"| CA1
    CA1 --> FFN1

    ExtractB -->|Query| CA2
    ExtractA -->|"Key / Value"| CA2
    CA2 --> FFN2

    %% 5. 聚合与分类
    subgraph Prediction_Head [预测头]
        Pool["Masked Mean Pooling<br>All 4 Features"]
        Concat["Concatenation<br>Dim: 1280*4"]
        MLP["Classifier MLP<br>Lin->BN->GELU->Prob"]
    end

    %% 连接到池化层 (原始特征 + 交互特征)
    ExtractA -.-> Pool
    ExtractB -.-> Pool
    FFN1 --> Pool
    FFN2 --> Pool

    Pool --> Concat
    Concat --> MLP
    MLP --> Output(("Interaction<br>Score"))

    %% 应用样式
    class InA,InB,Output input
    class ESM encoder
    class ExtractA,ExtractB process
    class CA1,FFN1,CA2,FFN2 decoder
    class Pool,Concat,MLP head
```

# 三, 训练结果

In [ ]:
# tensorboard --logdir /mnt/sdb/zht/IDRInter/logs/